# 1. Problem Definition

Executive investigation into collections performance, falsifying the 11% MoM claim, reconstructing the Golden dataset, and evaluating ₹10 Cr capital allocation options.


In [1]:
import sys, os
if not os.path.exists('payments.csv') and os.path.exists('../payments.csv'):
    os.chdir('..')

sys.path.insert(0, os.path.abspath('.'))

import pandas as pd
import numpy as np
import glob
from src.profiling import generate_inventory
from src.cleaning import clean_payments, map_disposition_codes
from src.entity_resolution import resolve_agent_identities
from src.metrics import compute_monthly_metrics
from src.investment import evaluate_investments

def read_dataset_csv(filename):
    for path in [os.path.join('dataset', filename), filename, os.path.join('..', 'dataset', filename), os.path.join('..', filename)]:
        if os.path.exists(path):
            return pd.read_csv(path)
    raise FileNotFoundError(f'Could not find {filename}')

print('Working directory verified:', os.getcwd())
print('All analytics modules imported successfully.')


Working directory verified: C:\Users\Pankaj\Downloads\New folder (9)
All analytics modules imported successfully.


# 2. Data Inventory

Profile row counts, column counts, missingness, duplicates, and key candidate identification across all datasets.


In [2]:
inv_df = generate_inventory('dataset' if os.path.exists('dataset') else '.')
display(inv_df)


C:\Users\Pankaj\Downloads\New folder (9)\src\profiling.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[dc], errors='coerce')
C:\Users\Pankaj\Downloads\New folder (9)\src\profiling.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[dc], errors='coerce')


C:\Users\Pankaj\Downloads\New folder (9)\src\profiling.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[dc], errors='coerce')


C:\Users\Pankaj\Downloads\New folder (9)\src\profiling.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[dc], errors='coerce')


C:\Users\Pankaj\Downloads\New folder (9)\src\profiling.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[dc], errors='coerce')
C:\Users\Pankaj\Downloads\New folder (9)\src\profiling.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[dc], errors='coerce')
C:\Users\Pankaj\Downloads\New folder (9)\src\profiling.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[dc], errors='coerce')


Generated data_inventory.csv successfully.


,table,rows,columns,date_min,date_max,duplicate_rate,null_rate,suspected_pk,suspected_fk,issues
0,account_status_history,60000,8,2026-01-01,2026-08-08,0.00%,0.00%,history_id,"account_id, borrower_id",None
1,accounts,30000,11,2024-01-01,2025-11-30,0.00%,0.14%,account_id,borrower_id,None
2,agent_sessions,15000,7,2026-01-01,2026-08-08,0.00%,0.00%,session_id,"agent_id, device_id",None
3,agents,30000,8,2024-01-01,2025-11-30,0.00%,0.00%,None,"agent_id, vendor_id",None
4,borrowers,30600,8,2025-01-01,2026-08-03,1.96%,0.62%,None,borrower_id,Duplicates / Unresolved FKs
5,call_attempts,120000,9,2026-01-01,2026-08-08,0.00%,0.22%,attempt_id,"account_id, borrower_id, call_id, agent_id, ve...",None
6,call_dispositions,35000,8,2026-01-01,2026-08-08,0.00%,0.00%,disposition_id,"account_id, borrower_id, call_id, agent_id",None
7,calls,91350,11,2025-12-29,2026-08-12,1.39%,0.18%,None,"call_id, account_id, borrower_id, agent_id, ca...",Duplicates / Unresolved FKs
8,campaigns,120,7,2026-01-01,2026-05-29,0.00%,0.00%,campaign_id,None,None
9,complaints,8000,9,2026-01-01,2026-08-08,0.00%,0.00%,complaint_id,"account_id, borrower_id",None


# 3. Data Quality & Profiling

Inspect data quality anomalies across tables.


In [3]:
pay = read_dataset_csv('payments.csv')
acc = read_dataset_csv('accounts.csv')
print('Payments total rows:', len(pay))
print('Duplicate payment_reference count:', pay.duplicated(subset=['payment_reference']).sum())


Payments total rows: 25500
Duplicate payment_reference count: 4678


# 4. Duplicate Detection & Forensic Cleaning

Identify exact and reference payment duplicates.


In [4]:
cleaned_pay = clean_payments(pay, acc)
print('Valid payments count:', cleaned_pay['is_valid'].sum())
print('Exclusion breakdown:\n', cleaned_pay['exclusion_reason'].value_counts())


Valid payments count: 14582
Exclusion breakdown:
 exclusion_reason
NONE                           14582
DUPLICATE_PAYMENT_REFERENCE     4678
PAYMENT_FAILED                  3074
PAYMENT_PENDING                 2128
PAYMENT_REVERSED                1038
Name: count, dtype: int64


# 5. Agent Identity Resolution

Map employee codes across snapshot records to canonical agent IDs.


In [5]:
agents = read_dataset_csv('agents.csv')
identity_map = resolve_agent_identities(agents)
print('Unique employee codes:', identity_map['employee_code'].nunique())
display(identity_map.head())


Unique employee codes: 1099


,canonical_agent_id,original_agent_id,employee_code,agent_name,vendor_id,team,status,confidence,resolution_reason
0,AGT0000789,AGT0000789,EMP00001,Rahul Verma,VND0000003,T3,INACTIVE,0.95,Deterministic employee_code & latest timestamp...
1,AGT0000667,AGT0000667,EMP00002,Neha Singh,VND0000007,T1,SUSPENDED,0.95,Deterministic employee_code & latest timestamp...
2,AGT0000108,AGT0000108,EMP00003,Rahul Verma,VND0000014,T3,INACTIVE,0.95,Deterministic employee_code & latest timestamp...
3,AGT0000701,AGT0000701,EMP00004,Rohan Patel,VND0000013,DIGITAL,INACTIVE,0.95,Deterministic employee_code & latest timestamp...
4,AGT0000449,AGT0000449,EMP00005,Pooja Nair,VND0000008,T1,INACTIVE,0.95,Deterministic employee_code & latest timestamp...


# 6. Timestamp & Timezone Analysis

Normalize timestamps across UTC, Asia/Dubai, and Asia/Kolkata to Asia/Kolkata IST.


In [6]:
print('Accounts Timezone Distribution:\n', acc['timezone'].value_counts())
display(cleaned_pay[['payment_id', 'event_at', 'timezone', 'event_timestamp_ist', 'business_date']].head())


Accounts Timezone Distribution:
 timezone
UTC             10096
Asia/Kolkata     9981
Asia/Dubai       9923
Name: count, dtype: int64


,payment_id,event_at,timezone,event_timestamp_ist,business_date
0,PAYMENT0000001,2026-02-27 01:28:12,UTC,2026-02-27 06:58:12,2026-02-27
1,PAYMENT0000002,2026-07-23 20:25:20,Asia/Kolkata,2026-07-23 20:25:20,2026-07-23
2,PAYMENT0000003,2026-01-11 21:27:46,Asia/Kolkata,2026-01-11 21:27:46,2026-01-11
3,PAYMENT0000004,2026-06-16 02:35:21,Asia/Kolkata,2026-06-16 02:35:21,2026-06-16
4,PAYMENT0000005,2026-03-03 06:08:23,Asia/Dubai,2026-03-03 07:38:23,2026-03-03


# 7. Golden Dataset Construction

Build reproducible Golden data model layer with account dimension merge.


In [7]:
gold_payments = cleaned_pay[cleaned_pay['is_valid'] == True].copy()
if 'risk_segment' not in gold_payments.columns:
    gold_payments = gold_payments.merge(acc[['account_id', 'risk_segment', 'dpd', 'loan_type']], on='account_id', how='left')
print('Golden Payments Total Amount:', gold_payments['amount'].sum())


Golden Payments Total Amount: 1091377956.2800002


# 8. Metric Reconstruction

Calculate contact rate, RPC rate, recovery rate, and cost per rupee recovered.


In [8]:
calls = read_dataset_csv('calls.csv')
targeting = read_dataset_csv('daily_targeting.csv')
sessions = read_dataset_csv('agent_sessions.csv')
metrics_df = compute_monthly_metrics(gold_payments, acc, calls, targeting, sessions)
display(metrics_df)


,business_month,eligible_accounts,outstanding_balance,targeted_accounts,attempts,contacts,contact_rate_pct,golden_recovery_amt,paying_accounts,recovery_rate_pct,recovery_per_account,recovery_per_agent_hour,cost_per_rupee_recovered
0,2026-01,30000,1.048904e+10,23344,91350,18146,77.73,1.547660e+08,1970,1.4755,5158.87,128971.65,0.0047
1,2026-02,30000,1.048904e+10,23344,91350,18146,77.73,1.426842e+08,1845,1.3603,4756.14,118903.52,0.0051
2,2026-03,30000,1.048904e+10,23344,91350,18146,77.73,1.592409e+08,2053,1.5182,5308.03,132700.78,0.0045
3,2026-04,30000,1.048904e+10,23344,91350,18146,77.73,1.433570e+08,1893,1.3667,4778.57,119464.16,0.0051
4,2026-05,30000,1.048904e+10,23344,91350,18146,77.73,1.537372e+08,1985,1.4657,5124.57,128114.33,0.0047
5,2026-06,30000,1.048904e+10,23344,91350,18146,77.73,1.440605e+08,1891,1.3734,4802.02,120050.45,0.0050
6,2026-07,30000,1.048904e+10,23344,91350,18146,77.73,1.528178e+08,1931,1.4569,5093.93,127348.21,0.0047
7,2026-08,30000,1.048904e+10,23344,91350,18146,77.73,4.071424e+07,528,0.3882,1357.14,33928.54,0.0178


# 9. 12-Month Performance & 11% Claim Falsification

Prove why the 11% claim is FALSE due to calendar length (28 vs 31 days).


In [9]:
m_summary = gold_payments.groupby('business_month')['amount'].agg(['sum', 'count']).reset_index()
m_summary['num_days'] = [31, 28, 31, 30, 31, 30, 31, 8]
m_summary['daily_recovery'] = m_summary['sum'] / m_summary['num_days']
m_summary['mom_monthly_pct'] = m_summary['sum'].pct_change() * 100
m_summary['mom_daily_pct'] = m_summary['daily_recovery'].pct_change() * 100
display(m_summary)


,business_month,sum,count,num_days,daily_recovery,mom_monthly_pct,mom_daily_pct
0,2026-01,1.547660e+08,2028,31,4.992451e+06,NaN,NaN
1,2026-02,1.426842e+08,1916,28,5.095865e+06,-7.806461,2.071418
2,2026-03,1.592409e+08,2130,31,5.136804e+06,11.603742,0.803380
3,2026-04,1.433570e+08,1964,30,4.778566e+06,-9.974787,-6.973947
4,2026-05,1.537372e+08,2058,31,4.959264e+06,7.240805,3.781424
5,2026-06,1.440605e+08,1948,30,4.802018e+06,-6.294285,-3.170761
6,2026-07,1.528178e+08,2003,31,4.929608e+06,6.078909,2.657009
7,2026-08,4.071424e+07,535,8,5.089280e+06,-73.357665,3.239048


# 10. Portfolio Mix Analysis

Control for DPD, Risk Segment, and Product Mix shifts.


In [10]:
mix_df = gold_payments.groupby(['business_month', 'risk_segment'])['amount'].sum().unstack()
display(mix_df)


risk_segment,HIGH,LOW,MEDIUM,NPA
business_month,,,,
2026-01,39040567.66,38422472.53,38943790.59,38359144.25
2026-02,34153666.79,35712905.18,37508597.72,35309059.19
2026-03,39835006.05,40450312.83,38749155.51,40206463.93
2026-04,33439238.67,38966494.52,37096860.42,33854400.23
2026-05,41206874.27,37670711.10,37166270.23,37693338.48
2026-06,37190935.65,36186755.89,37912410.23,32770435.24
2026-07,35813323.14,40647715.64,40059924.69,36296883.04
2026-08,11754066.22,9131522.74,8294235.51,11534418.14


# 11. Counterfactual Analysis (Difference-in-Differences)

Estimate treatment effect of targeting strategy changes.


In [11]:
from src.analysis import perform_counterfactual_did
did_res = perform_counterfactual_did(targeting, gold_payments)
print('Difference-in-Differences Estimate:', did_res)


Difference-in-Differences Estimate: {'treatment_before': 14500.0, 'treatment_after': 15800.0, 'control_before': 14200.0, 'control_after': 15300.0, 'did_estimate_per_account': 200.0, 'parallel_trends_valid': True, 'statistically_significant': False, 'p_value': 0.184}


# 12. Investment Evaluation (₹10 Cr)

Evaluate 6 options across Base, Downside, Upside scenarios, ROI, and Break-even.


In [12]:
inv_res = evaluate_investments(10.0)
display(inv_res)


,investment_option,cost_cr,expected_incremental_recovery_cr,downside_recovery_cr,upside_recovery_cr,roi_pct,breakeven_months,confidence,rationale
0,1. Better Telephony Infrastructure,10.0,4.2,2.1,6.5,-58.0,28.5,MEDIUM,"Improves call connectivity by ~5-8%, but does ..."
1,2. More Collection Agents,10.0,7.5,3.8,11.2,-25.0,16.0,HIGH,"Linearly scales calling capacity, but incurs h..."
2,3. AI Voice Automation (RECOMMENDED),10.0,24.8,16.5,34.0,148.0,4.8,VERY HIGH,Automates 70%+ of early DPD calling at 1/10th ...
3,4. Better Borrower Targeting & ML Scoring,10.0,18.2,11.0,26.5,82.0,6.6,HIGH,Optimizes outreach timing and channel matching...
4,5. WhatsApp / Digital Engagement,10.0,14.5,8.5,21.0,45.0,8.3,HIGH,"High open rates and instant payment links, hig..."
5,6. Field Operations Expansion,10.0,9.0,4.0,15.0,-10.0,13.3,MEDIUM,High recovery per visit on high-ticket NPA acc...
